# Tech Challenge Fase 2 - Previsao de Tendencia do IBOVESPA

Projeto desenvolvido para prever se o fechamento do IBOVESPA no proximo pregao sera maior ou menor que o fechamento do pregao atual.

A proposta segue o enunciado da atividade: usar dados historicos diarios do IBOVESPA, reservar os ultimos 30 pregoes disponiveis para teste e avaliar se o modelo atinge pelo menos 75% de acuracia.

## 1. Preparacao do ambiente

Este notebook foi organizado para rodar tanto no Google Colab quanto em ambiente local. No Colab, o repositorio publico sera clonado automaticamente antes da execucao.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/renancotrin-alt/tech-challenge-fase-2-ibovespa.git"
REPO_BRANCH = "V1_Ibovespa"
PROJECT_DIR = Path("tech-challenge-fase-2-ibovespa")

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    if not PROJECT_DIR.exists():
        subprocess.run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL], check=True)
    os.chdir(PROJECT_DIR)
else:
    if Path.cwd().name == "notebooks":
        os.chdir("..")

print("Diretorio de trabalho:", Path.cwd())

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

from src.data_prep import build_modeling_dataset, load_raw_data, save_processed_dataset, temporal_train_test_split
from src.modeling import FEATURE_COLUMNS, TARGET_COLUMN

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 80)

## 2. Aquisicao e entendimento dos dados

A base foi obtida no Investing.com com periodicidade diaria. O arquivo original esta em `data/raw/dados_historicos_ibovespa.csv` e mantem o formato brasileiro de numeros, como `133.990` para representar 133.990 pontos.

In [ ]:
raw_df = load_raw_data()
print("Linhas e colunas da base original:", raw_df.shape)
display(raw_df.head())
display(raw_df.tail())

## 3. Limpeza, target e engenharia de atributos

A preparacao transforma os campos numericos, ordena os pregoes cronologicamente e cria o target `alta_amanha`.

O target recebe valor `1` quando o fechamento do proximo pregao e maior que o fechamento atual; caso contrario, recebe `0`.

As features foram construidas apenas com informacoes conhecidas ate o dia da previsao, reduzindo risco de vazamento temporal.

In [ ]:
df = build_modeling_dataset()
save_processed_dataset(df)

print("Base de modelagem:", df.shape)
print("Periodo:", df["data"].min().date(), "ate", df["data"].max().date())
print("Proporcao de altas no total:", f"{df[TARGET_COLUMN].mean():.2%}")
display(df.head())

As principais familias de atributos criadas foram:

- retornos acumulados e defasados;
- medias moveis e distancia em relacao as medias;
- volatilidade recente;
- amplitude intradiaria e corpo do candle;
- posicao do fechamento dentro de janelas recentes;
- indicadores tecnicos simples, como RSI, MACD e posicao nas Bandas de Bollinger.

## 4. Analise exploratoria

Antes da modelagem, observamos a trajetoria do indice, a distribuicao dos retornos diarios e o equilibrio entre as classes do target.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

sns.lineplot(data=df, x="data", y="fechamento", ax=axes[0], color="#1f77b4")
axes[0].set_title("Fechamento do IBOVESPA")
axes[0].set_xlabel("Data")
axes[0].set_ylabel("Pontos")

sns.histplot(df["retorno_1d"], bins=40, kde=True, ax=axes[1], color="#2ca02c")
axes[1].set_title("Distribuicao dos retornos diarios")
axes[1].set_xlabel("Retorno diario")
axes[1].set_ylabel("Frequencia")

plt.tight_layout()
plt.show()

In [ ]:
classe_df = (
    df[TARGET_COLUMN]
    .value_counts(normalize=True)
    .rename(index={0: "Baixa/igual", 1: "Alta"})
    .mul(100)
    .reset_index()
)
classe_df.columns = ["classe", "percentual"]

display(classe_df)
sns.barplot(data=classe_df, x="classe", y="percentual", hue="classe", palette=["#d62728", "#2ca02c"], legend=False)
plt.title("Distribuicao do target")
plt.ylabel("Percentual")
plt.xlabel("")
plt.ylim(0, 100)
plt.show()

## 5. Separacao temporal entre treino e teste

Como se trata de serie temporal, a divisao nao pode ser aleatoria. O conjunto de teste foi formado pelos ultimos 30 pregoes disponiveis na base, simulando uma avaliacao em periodo futuro.

In [ ]:
train_df, test_df = temporal_train_test_split(df, test_size=30)

print("Treino:", len(train_df), train_df["data"].min().date(), "ate", train_df["data"].max().date())
print("Teste:", len(test_df), test_df["data"].min().date(), "ate", test_df["data"].max().date())
print("Proporcao de altas no teste:", f"{test_df[TARGET_COLUMN].mean():.2%}")

## 6. Modelagem

Foram comparados dois baselines e quatro modelos de classificacao. O modelo final usa uma janela recente de 150 pregoes, pois o mercado financeiro muda de regime ao longo do tempo; dar peso excessivo a periodos antigos pode reduzir a capacidade de capturar o comportamento atual.

In [ ]:
class MajorityClassBaseline:
    def fit(self, _x, y):
        self.majority_class_ = int(y.mode().iloc[0])
        return self

    def predict(self, x):
        return [self.majority_class_] * len(x)


class LastTrendBaseline:
    def fit(self, _x, y):
        self.last_class_ = int(y.iloc[-1])
        return self

    def predict(self, x):
        return [self.last_class_] * len(x)


model_specs = [
    ("Baseline - classe majoritaria", MajorityClassBaseline(), None),
    ("Baseline - ultima tendencia", LastTrendBaseline(), None),
    (
        "Regressao Logistica",
        Pipeline([
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(max_iter=1000, class_weight="balanced", C=0.1, random_state=42)),
        ]),
        None,
    ),
    (
        "Random Forest",
        RandomForestClassifier(n_estimators=300, max_depth=4, min_samples_leaf=8, class_weight="balanced", random_state=42),
        None,
    ),
    (
        "Gradient Boosting",
        GradientBoostingClassifier(n_estimators=120, learning_rate=0.04, max_depth=2, min_samples_leaf=8, random_state=42),
        None,
    ),
    (
        "SVC - janela recente 150 pregoes",
        Pipeline([
            ("scaler", StandardScaler()),
            ("model", SVC(C=1.0, kernel="rbf", class_weight="balanced", random_state=42)),
        ]),
        150,
    ),
]

In [ ]:
x_test = test_df[FEATURE_COLUMNS]
y_test = test_df[TARGET_COLUMN]

results = []
predictions = {}

for name, model, window_size in model_specs:
    model_train_df = train_df if window_size is None else train_df.tail(window_size)
    x_train = model_train_df[FEATURE_COLUMNS]
    y_train = model_train_df[TARGET_COLUMN]

    model.fit(x_train, y_train)
    y_pred = np.array(model.predict(x_test))
    predictions[name] = y_pred

    cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
    results.append({
        "modelo": name,
        "janela_treino": len(model_train_df),
        "acuracia": accuracy_score(y_test, y_pred),
        "tn_baixa_correta": cm[0, 0],
        "fp_alta_errada": cm[0, 1],
        "fn_baixa_errada": cm[1, 0],
        "tp_alta_correta": cm[1, 1],
    })

results_df = pd.DataFrame(results).sort_values("acuracia", ascending=False)
results_df["acuracia"] = results_df["acuracia"].map(lambda value: f"{value:.2%}")
display(results_df)

## 7. Resultado final

O melhor desempenho no conjunto de teste foi obtido pelo SVC treinado com os 150 pregoes mais recentes do conjunto de treino.

In [ ]:
final_model_name = "SVC - janela recente 150 pregoes"
final_pred = predictions[final_model_name]

print("Modelo final:", final_model_name)
print("Acuracia final:", f"{accuracy_score(y_test, final_pred):.2%}")
print()
print(classification_report(y_test, final_pred, target_names=["Baixa/igual", "Alta"], zero_division=0))

ConfusionMatrixDisplay.from_predictions(
    y_test,
    final_pred,
    display_labels=["Baixa/igual", "Alta"],
    cmap="Blues",
    colorbar=False,
)
plt.title("Matriz de confusao - modelo final")
plt.show()

In [ ]:
comparison_df = test_df[["data", "fechamento", "fechamento_amanha", TARGET_COLUMN]].copy()
comparison_df["previsao_alta"] = final_pred
comparison_df["acertou"] = comparison_df[TARGET_COLUMN] == comparison_df["previsao_alta"]
comparison_df["classe_real"] = comparison_df[TARGET_COLUMN].map({0: "Baixa/igual", 1: "Alta"})
comparison_df["classe_prevista"] = comparison_df["previsao_alta"].map({0: "Baixa/igual", 1: "Alta"})

display(comparison_df[["data", "fechamento", "fechamento_amanha", "classe_real", "classe_prevista", "acertou"]])

## 8. Conclusao gerencial

O modelo final atingiu acuracia acima do minimo solicitado no teste temporal dos ultimos 30 pregoes. O resultado indica que os indicadores recentes do proprio IBOVESPA carregam algum sinal util para apoiar a leitura de tendencia do proximo pregao.

Mesmo assim, a previsao deve ser entendida como apoio analitico, nao como recomendacao automatica de compra ou venda. Mercados financeiros sao ruidosos e sensiveis a eventos externos; por isso, o modelo deve ser usado junto com outros indicadores e com revisao periodica.